# Required assignment 9.1 Coding your decision trees

In this notebook, you’ll build a decision tree classifier using a real-world data set in Python. The workflow starts with thorough data preprocessing, followed by model training, evaluation and visualisation. You’ll then impute additional data sets to further test the classifier and analyse the resulting confusion matrix.

In [ ]:
### Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

from sklearn.tree import plot_tree


The [Telco Customer Churn](https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset) data set contains information about 7,043 customers of a fictional telecommunications company, including demographics, account details, services subscribed and whether the customer has churned or not. This data set is widely used for building predictive models to identify customers at risk of leaving, enabling organisations to develop effective retention strategies.


In [ ]:
df = pd.read_csv('data/Telco_customer_churn.csv')

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isna().sum()

## Data preprocessing

When building a machine learning model to predict customer churn (or a similar outcome), not all columns in the data set are useful. Some should be excluded because they serve as unique identifiers, are redundant, are overly granular or are directly related to the target variable. For example, columns such as `CustomerID`, `City`, `State`, `Zip Code`, `Lat Long`, `Latitude`, `Longitude`, `Churn Value`, `Churn Score` and `Churn Reason` can be dropped, as they either provide limited predictive value or leak information about the outcome.


### Question 1

Drop the columns listed in `columns_to_drop` from the data set `df`, and store the resulting data frame in `df_cleaned`.

In [ ]:
### GRADED CELL
columns_to_drop = [
    'CustomerID', 'City', 'State', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude',
    'Churn Label', 'Churn Score', 'Churn Reason'
]
df_cleaned = ...
# Drop the columns
### BEGIN SOLUTION
df_cleaned = df.drop(columns=columns_to_drop, errors = 'ignore')
### END SOLUTION
# Display the remaining columns
print("Remaining columns:")
print(df_cleaned.columns)

### Question 2

The target is `Churn Value`.
Find the unique values of `Churn Value` and assign it to `ans2`.

In [ ]:
### GRADED

ans2 = None

### BEGIN SOLUTION
ans2 = df_cleaned.get('Churn Value').unique()
### END SOLUTION

print("The unique values of `Churn Value` is given by")
print(ans2)


The first step in data preprocessing is to remove columns that are not relevant for model building. 

Next, you eliminate features that have little correlation with the target variable. To do this, you’ll first explore some of the columns and select the most appropriate ones by choosing a mix of numeric and non-numeric features.

Let’s begin by separating the target variable from the input features.

In [ ]:
df_cleaned.columns


### Question 3

From the `df_cleaned` data set, separate the input features and the target variable `Churn Value`:

- Drop the `Churn Value` from the input features in the `df_cleaned`data set and store the features in `X`.

- Store the target `Churn Value` in the output `y`.

In [ ]:
### GRADED CELL
X=None
y=None

### BEGIN SOLUTION
X = df_cleaned.drop(columns=['Churn Value'])
y = df_cleaned['Churn Value']
### END SOLUTION
### Answer test

print(X.shape)
print(y.shape)


The features in `selected_features` are chosen for their relevance to churn behaviour. They reflect customer relationship duration, payment and contract flexibility, service usage and demographic risk factors. These factors are commonly linked to churn in both research and industry practice.

The `selected_features` has both numeric and categorical features.

### Question 4

- Separate the numeric and categorical features.

- Encode the categorical features (use `get_dummies` for one-hot encoding).

- Concatenate the numeric and encoded categorical features.

Note: `Tenure Months` and `Total Charges` are the only numeric features.

In [ ]:
### GRADED CELL

selected_features = [
    'Tenure Months', 'Total Charges', 'Contract',
    'Payment Method', 'Paperless Billing', 'Internet Service',
   'Tech Support', 'Streaming TV', 'Streaming Movies', 'Senior Citizen', 'Gender'
    ]
numeric_features = ...
categorical_features = ...
X = ...
df_numeric = ...
df_categorical = ...
### BEGIN SOLUTION
numeric_features = ['Tenure Months', 'Total Charges']
categorical_features = [
    'Contract', 'Payment Method', 'Paperless Billing', 'Internet Service',
    'Tech Support', 'Streaming TV', 'Streaming Movies', 'Senior Citizen', 'Gender'
]
# 2. Encode the categorical features (using get_dummies for one-hot encoding)
df_numeric = df[numeric_features]
df_categorical = pd.get_dummies(df[categorical_features], drop_first=True)

# 3. Concatenate the numeric and encoded categorical features
X = pd.concat([df_numeric, df_categorical], axis=1)
### END SOLUTION
# Answer test
print(X.shape)


In [ ]:
X.replace(r'^\s*$', np.nan, regex=True, inplace=True)
X = X.infer_objects()
X.fillna(0, inplace=True)
X = X.apply(pd.to_numeric)

### Question 5

- Use `train_test_split` from `sklearn.model_selection` and split the data into`X_train`,`X_test`,`y_train`,`y_test`. Use 20 per cent as the`test_size`.

- Model a `DecisionTreeClassifier` and store the model in `clf`.

- Fit the model and compute `y_pred` using the `.predict()` function.

In [ ]:
### GRADED CELL
X_train, X_test, y_train, y_test = None, None, None, None
clf = None
y_pred = None

### BEGIN SOLUTION
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
### END SOLUTION
# Answer test
print(y_pred)


### Question 6

- Compute the `confusion_matrix` using the `y_test` and `y_pred` functions.

- Compute the accuracy.

In [ ]:
### GRADED CELL
cm = None
accuracy = None

### BEGIN SOLUTION
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
### END SOLUTION
# Answer test

print("Confusion Matrix:\n", cm)

print(f"Accuracy: {accuracy:.2f}")


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Pre-pruned decision tree (limits depth and leaf size)
clf_pruned = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,           # Limit tree depth
    min_samples_leaf=5,    # Minimum samples per leaf
    max_leaf_nodes=10,     # Maximum number of leaf nodes
    random_state=42
)
clf_pruned.fit(X_train, y_train)


In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))  # Adjust size as needed
plot_tree(
    clf_pruned,
    feature_names=X_train.columns,         # Use your feature names
    class_names=['No Churn', 'Churn'],     # Adjust if your target classes are different
    filled=True,
    rounded=True
)
plt.show()


By applying pre-pruning techniques and visualising the resulting tree, this approach strikes a balance between interpretability and performance, keeping the decision tree both understandable and less prone to overfitting.